In [ ]:
%pip install unitycatalog-ai[databricks] databricks-vectorsearch -q
dbutils.library.restartPython()

# Pattern ② HTTP API連携

DifyのHTTPリクエストノードを使って、Databricks REST APIを直接呼び出します。

## Databricksが提供するAPI一覧

| API | エンドポイント | 形式 | Dify連携 |
|-----|-------------|------|----------|
| UC Functions | SQL Statements API | REST (同期) | ◎ HTTPノードで直接呼べる |
| Vector Search | Vector Search API | REST (同期) | ◎ HTTPノードで直接呼べる |
| Genie | Conversation API | REST (非同期) | △ ポーリング必要 |
| KA / MAS / Agent | Serving Endpoints | ResponsesAgent形式 | △ input/output形式 |

## アーキテクチャ
```
Dify Workflow
  │
  ├── HTTPリクエスト ──▶ SQL Statements API（UC関数実行）
  ├── HTTPリクエスト ──▶ Vector Search API（セマンティック検索）
  ├── HTTPリクエスト ──▶ Genie API（自然言語→SQL）
  └── HTTPリクエスト ──▶ Serving Endpoints（KA/MAS/Agent呼び出し）
```

In [ ]:
%run ./_config

In [ ]:
import requests, json
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}
print(f"Workspace: https://{host}")

## Databricksリソースの構築

HTTP API連携のテストに必要なUC関数とVector Searchリソースを作成します。

### UC関数の作成

In [ ]:
# UC関数1: 顧客情報の取得
spark.sql(f"""
CREATE OR REPLACE FUNCTION `{catalog}`.`{schema}`.get_customer_by_email(email_input STRING COMMENT '顧客のメールアドレス')
RETURNS TABLE (
    customer_id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    phone STRING,
    prefecture STRING,
    customer_status STRING,
    loyalty_tier STRING
)
COMMENT 'メールアドレスで顧客情報を検索します。顧客ID、氏名、連絡先、会員ランク等を返します。'
RETURN (
    SELECT customer_id, first_name, last_name, email, phone, prefecture, customer_status, loyalty_tier
    FROM `{catalog}`.`{schema}`.customers
    WHERE email = email_input
    LIMIT 1
)
""")
print("✅ get_customer_by_email 作成完了")

In [ ]:
# UC関数2: 注文履歴の取得
spark.sql(f"""
CREATE OR REPLACE FUNCTION `{catalog}`.`{schema}`.get_order_history(customer_id_input BIGINT COMMENT '顧客ID')
RETURNS TABLE (
    order_id BIGINT,
    product_name STRING,
    quantity INT,
    total_amount BIGINT,
    order_date STRING,
    status STRING
)
COMMENT '顧客IDに基づいて注文履歴を取得します。直近の注文から順に返します。'
RETURN (
    SELECT order_id, product_name, quantity, total_amount, order_date, status
    FROM `{catalog}`.`{schema}`.orders
    WHERE customer_id = customer_id_input
    ORDER BY order_date DESC
    LIMIT 10
)
""")
print("✅ get_order_history 作成完了")

In [ ]:
# UC関数3: サポートチケットの取得
spark.sql(f"""
CREATE OR REPLACE FUNCTION `{catalog}`.`{schema}`.get_support_tickets(customer_id_input BIGINT COMMENT '顧客ID')
RETURNS TABLE (
    ticket_id BIGINT,
    product_name STRING,
    category STRING,
    priority STRING,
    status STRING,
    description STRING,
    created_date STRING
)
COMMENT '顧客IDに基づいてサポートチケット一覧を取得します。'
RETURN (
    SELECT ticket_id, product_name, category, priority, status, description, created_date
    FROM `{catalog}`.`{schema}`.support_tickets
    WHERE customer_id = customer_id_input
    ORDER BY created_date DESC
    LIMIT 10
)
""")
print("✅ get_support_tickets 作成完了")

In [ ]:
# UC関数4: ポリシー検索
spark.sql(f"""
CREATE OR REPLACE FUNCTION `{catalog}`.`{schema}`.search_policy(keyword STRING COMMENT '検索キーワード（例: 返品、保証、ポイント）')
RETURNS TABLE (
    policy_name STRING,
    category STRING,
    content STRING
)
COMMENT 'キーワードに基づいて社内ポリシーを検索します。'
RETURN (
    SELECT policy_name, category, content
    FROM `{catalog}`.`{schema}`.policies
    WHERE lower(content) LIKE concat('%', lower(keyword), '%')
       OR lower(policy_name) LIKE concat('%', lower(keyword), '%')
       OR lower(category) LIKE concat('%', lower(keyword), '%')
)
""")
print("✅ search_policy 作成完了")

In [ ]:
# UC関数5: 数学計算（Python UDF）
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

def calculate_math_expression(expression: str) -> float:
    """
    数学式を安全に評価します。

    Args:
        expression (str): 数学式（例: "sqrt(2 + 3 * (4 - 1))"、Pythonのmath関数を使用可能）

    Returns:
        float: 計算結果
    """
    import math
    allowed_names = {k: v for k, v in math.__dict__.items() if not k.startswith("__")}
    allowed_names.update({"abs": abs, "round": round})
    try:
        result = eval(expression, {"__builtins__": None}, allowed_names)
        return float(result)
    except Exception as e:
        raise ValueError(f"無効な数式: {expression}. エラー: {str(e)}")

client = DatabricksFunctionClient()
client.create_python_function(
    func=calculate_math_expression,
    catalog=catalog,
    schema=schema,
    replace=True
)
print("✅ calculate_math_expression 作成完了")

### Vector Search Endpoint & Index の作成

In [ ]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

# Endpoint作成（既に存在する場合はスキップ）
try:
    vsc.get_endpoint(name=VECTOR_SEARCH_ENDPOINT_NAME)
    print(f"✅ Vector Search Endpoint '{VECTOR_SEARCH_ENDPOINT_NAME}' は既に存在します")
except Exception:
    vsc.create_endpoint(name=VECTOR_SEARCH_ENDPOINT_NAME, endpoint_type="STANDARD")
    print(f"⏳ Vector Search Endpoint '{VECTOR_SEARCH_ENDPOINT_NAME}' を作成中...")

In [ ]:
vs_index_fullname = f"{catalog}.{schema}.knowledge_base_vs_index"
source_table = f"{catalog}.{schema}.knowledge_base"

# Index作成（既に存在する場合はスキップ）
try:
    vsc.get_index(VECTOR_SEARCH_ENDPOINT_NAME, vs_index_fullname)
    print(f"✅ Vector Search Index '{vs_index_fullname}' は既に存在します")
except Exception:
    vsc.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=vs_index_fullname,
        source_table_name=source_table,
        pipeline_type="TRIGGERED",
        primary_key="id",
        embedding_source_column="content",
        embedding_model_endpoint_name=EMBEDDING_MODEL_NAME,
    )
    print(f"⏳ Vector Search Index を作成中...")
print("✅ リソース作成セクション完了")

---
## 1. SQL Statements API（UC関数の実行）

UC関数をSQL経由で呼び出します。Difyからは最もシンプルに使えるパターンです。

In [ ]:
# ウェアハウスID取得（最初のServerless Warehouseを使用）
warehouses = requests.get(f"https://{host}/api/2.0/sql/warehouses", headers=headers).json()
warehouse_id = None
for wh in warehouses.get("warehouses", []):
    if wh.get("warehouse_type") == "PRO" or wh.get("enable_serverless_compute"):
        warehouse_id = wh["id"]
        break
if not warehouse_id:
    warehouse_id = warehouses.get("warehouses", [{}])[0].get("id", "")
print(f"Warehouse ID: {warehouse_id}")

In [ ]:
# UC関数をSQL経由で呼び出し
sql_url = f"https://{host}/api/2.0/sql/statements/"
sql_payload = {
    "warehouse_id": warehouse_id,
    "statement": f"SELECT * FROM {catalog}.{schema}.search_policy('返品')",
    "wait_timeout": "30s"
}

response = requests.post(sql_url, headers=headers, json=sql_payload)
result = response.json()

if result.get("status", {}).get("state") == "SUCCEEDED":
    columns = [c["name"] for c in result["manifest"]["schema"]["columns"]]
    print("✅ UC関数実行成功")
    print("カラム:", columns)
    for row in result["result"]["data_array"][:3]:
        print(dict(zip(columns, row)))
else:
    print("❌ エラー:", result)

In [ ]:
# Python UC関数のテスト（calculate_math_expression）
sql_payload_math = {
    "warehouse_id": warehouse_id,
    "statement": f"SELECT {catalog}.{schema}.calculate_math_expression('sqrt(2 + 3 * 4)')",
    "wait_timeout": "30s"
}

math_resp = requests.post(sql_url, headers=headers, json=sql_payload_math)
math_result = math_resp.json()

if math_result.get("status", {}).get("state") == "SUCCEEDED":
    value = math_result["result"]["data_array"][0][0]
    print(f"✅ Python UC関数: sqrt(2 + 3 * 4) = {value}")
else:
    print("❌ エラー:", math_result)

In [ ]:
# Dify HTTPリクエストノード設定値
print("=" * 60)
print("Dify HTTPリクエストノード設定（SQL API）")
print("=" * 60)
print(f"\n【Method】POST")
print(f"【URL】https://{host}/api/2.0/sql/statements/")
print(f"\n【認証】")
print(f"  タイプ: API Key")
print(f"  Auth type: Bearer")
print(f"  API Key: <PAT_TOKEN>")
print(f"\n【Headers】")
print(f"  Content-Type: application/json")
print(f"\n【Body (JSON)】")
body = {
    "warehouse_id": warehouse_id,
    "statement": f"SELECT * FROM {catalog}.{schema}.search_policy('{{{{#start.input#}}}}')",
    "wait_timeout": "30s"
}
print(json.dumps(body, indent=2, ensure_ascii=False))
print(f"\n【Difyワークフロー構成】")
print("  Start → Code（SQL構築）→ HTTP（API実行）→ LLM（結果整形）→ End")
print("  ※ 複数UC関数を切り替える場合、Codeノードで動的にSQL文を組み立てる")


---
## 2. Vector Search API（セマンティック検索）

Agent構築で作成したVector Searchインデックスに対してREST APIで検索します。

In [ ]:
vs_index_name = f"{catalog}.{schema}.knowledge_base_vs_index"
vs_api_url = f"https://{host}/api/2.0/vector-search/indexes/{vs_index_name}/query"

vs_payload = {
    "query_text": "SmartWatch X100のバッテリー持続時間",
    "columns": ["content", "doc_uri"],
    "num_results": 3
}

vs_response = requests.post(vs_api_url, headers=headers, json=vs_payload)
vs_result = vs_response.json()

if "result" in vs_result:
    print("✅ Vector Search API 応答:")
    for row in vs_result["result"].get("data_array", [])[:3]:
        print(f"  Score: {row[-1]:.3f}")
        print(f"  Content: {row[0][:100]}...")
        print()
else:
    print("❌ エラー:", vs_result)

In [ ]:
print("=" * 60)
print("Dify HTTPリクエストノード設定（Vector Search）")
print("=" * 60)
print(f"\n【Method】POST")
print(f"【URL】https://{host}/api/2.0/vector-search/indexes/{vs_index_name}/query")
print(f"\n【認証】")
print(f"  タイプ: API Key")
print(f"  Auth type: Bearer")
print(f"  API Key: <PAT_TOKEN>")
print(f"\n【Headers】")
print(f"  Content-Type: application/json")
print(f"\n【Body (JSON)】")
vs_body = {
    "query_text": "{{#start.input#}}",
    "columns": ["content", "doc_uri"],
    "num_results": 3
}
print(json.dumps(vs_body, indent=2, ensure_ascii=False))
print(f"\n【Difyワークフロー構成】")
print("  Start → HTTP（検索実行）→ LLM（結果整形）→ End")


### Genie Space の作成

Genie APIのテストにはGenie Spaceが必要です。

1. Databricks Workspace → **Genie** → **New** で新しいSpaceを作成
2. テーブルとして `{catalog}.{schema}.products`, `{catalog}.{schema}.orders` 等を追加
3. 作成後、URLから Space ID を取得（例: `https://<workspace>/genie/rooms/<SPACE_ID>`）
4. `config.yaml` の `genie_space_id` に設定

> Genie SpaceはREST APIでの作成も可能ですが、UI経由が簡単です。

In [ ]:
# Genie Space ID の確認
if not GENIE_SPACE_ID:
    print("⚠️ config.yaml の genie_space_id が未設定です。")
    print("  Genie APIのテストをスキップする場合は、次のセクションに進んでください。")
else:
    print(f"✅ Genie Space ID: {GENIE_SPACE_ID}")

---
## 3. Genie Conversation API（自然言語→SQL）

Genie SpaceはREST APIで外部から呼び出せます。  
**非同期API**のため、会話開始→ポーリング→結果取得の3ステップが必要です。

| ステップ | メソッド | パス |
|---------|---------|------|
| 会話開始 | POST | `/api/2.0/genie/spaces/{space_id}/start-conversation` |
| 結果確認 | GET | `/api/2.0/genie/spaces/{space_id}/conversations/{conv_id}/messages/{msg_id}` |
| データ取得 | GET | `...messages/{msg_id}/query-result/{attachment_id}` |

### Difyでの実装上の注意

| 課題 | 対策 |
|------|------|
| 非同期APIでポーリングが必要 | Codeノード内で `urllib.request` + `time.sleep` ループ |
| サンドボックスのデフォルトタイムアウトが15秒 | `.env` で `SANDBOX_WORKER_TIMEOUT=120` に変更 |
| サンドボックスからのSSL証明書エラー | `ssl.CERT_NONE` でスキップ |
| レート制限 | 5 POST/分/ワークスペース（GETはカウント外） |

**Difyワークフロー構成**:
```
Start → HTTP（会話開始）→ Code（ポーリング+結果抽出）→ End
```


In [ ]:
import time

if not GENIE_SPACE_ID:
    print("⚠️ GENIE_SPACE_ID が未設定のためスキップします")
else:
    # Step 1: 会話開始
    genie_url = f"https://{host}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/start-conversation"
    genie_resp = requests.post(genie_url, headers=headers, json={"content": "商品は何件ありますか？"})
    genie_data = genie_resp.json()

    conv_id = genie_data["conversation_id"]
    msg_id = genie_data["message_id"]
    print(f"✅ 会話開始: conv={conv_id}, msg={msg_id}")

    # Step 2: ポーリングで結果を待つ
    poll_url = f"https://{host}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/conversations/{conv_id}/messages/{msg_id}"
    for i in range(20):
        time.sleep(3)
        poll_resp = requests.get(poll_url, headers=headers).json()
        status = poll_resp.get("status", "UNKNOWN")
        print(f"  Polling {i+1}: {status}")
        if status in ["COMPLETED", "FAILED", "CANCELLED"]:
            break

    # Step 3: 結果表示
    if status == "COMPLETED":
        attachments = poll_resp.get("attachments", [])
        for att in attachments:
            text = att.get("text", "")
            if isinstance(text, dict):
                text = text.get("content", str(text))
            if text:
                print(f"\n✅ Genie回答: {str(text)[:300]}")
            query = att.get("query", {})
            if isinstance(query, dict) and query.get("query"):
                print(f"\n  生成SQL: {str(query['query'])[:200]}")
    else:
        print(f"❌ Genie応答失敗: {status}")

---
## 4. Serving Endpoints（KA / MAS / Agent）

KA/MAS/Agent Serving Endpoints も同様にHTTP APIで呼び出し可能です。ResponsesAgent形式（`input`/`output`）でリクエストを送信します。詳細は `dsl/optional/` のDSLファイルを参照してください。

```json
{
    "input": [{"role": "user", "content": "ユーザーの質問"}]
}
```

> **注意**: ResponsesAgent形式はOpenAI Chat Completions形式（`messages`/`choices`）とは異なります。
> DifyではHTTPリクエストノードで呼び出し、Codeノードで回答を抽出します。

---
## まとめ: Difyから使えるDatabricks API

| API | URL パターン | リクエスト形式 | Dify連携の容易さ |
|-----|------------|-------------|----------------|
| SQL API | `/api/2.0/sql/statements/` | `{statement, warehouse_id}` | ◎ 同期・シンプル |
| Vector Search | `/api/2.0/vector-search/indexes/.../query` | `{query_text, columns}` | ◎ 同期・シンプル |
| Genie | `/api/2.0/genie/spaces/.../start-conversation` | `{content}` | △ 非同期・ポーリング必要 |
| KA/MAS/Agent | `/serving-endpoints/.../invocations` | `{input: [{role, content}]}` | ○ 同期だがResponsesAgent形式 |

### Difyワークフロー構成（API別）

| API | ワークフロー構成 | LLMノード |
|-----|----------------|----------|
| SQL API | Start → Code（SQL構築）→ HTTP → LLM（整形）→ End | **必要**（生JSONを整形） |
| Vector Search | Start → HTTP → LLM（整形）→ End | **必要**（生JSONを整形） |
| Genie | Start → HTTP（開始）→ Code（ポーリング+抽出）→ End | 不要 |
| KA/MAS/Agent | Start → HTTP → Code（回答抽出）→ End | 不要（自然言語で返る） |

### Dify HTTPリクエストノードの認証設定

Headersに `Authorization: Bearer` を手動で書くのではなく、**認証セクション**を使う：
```
認証タイプ: API Key
Auth type:  Bearer
API Key:    <PAT_TOKEN>
```

### ポイント
- 全API共通で `Authorization: Bearer <PAT>` で認証
- UC関数 / Vector Search は**同期API**でDifyのHTTPノードと相性が良い
- KA / MAS / Code Agentは**ResponsesAgent形式**（`input`/`output`）— LLMプロバイダーではなくHTTPツールとして使う
- GenieはDifyのCodeノード内でポーリングを実装（`SANDBOX_WORKER_TIMEOUT=120`が必要）
- ResponsesAgentのレスポンスには `function_call`, `function_call_output`, `message` が混在 — 最終回答は `type=message` のみ


---
## 補足: 認証・権限管理の仕組み

### 全パターン共通

全APIは **PAT（Personal Access Token）** で認証します。PATからユーザー/SPを特定し、Unity Catalogで権限チェックします。

```
Dify HTTP Node → Bearer <PAT> → Databricks → PATでユーザー特定 → UC権限チェック
```

### パターン別の権限チェック

| パターン | 必要なもの | 権限チェック |
|---------|----------|------------|
| SQL API（UC関数） | PAT + warehouse_id | Warehouse CAN USE + UC EXECUTE FUNCTION |
| Vector Search | PAT | UC SELECT on index |
| Genie | PAT + space_id | Space CAN VIEW |
| KA/MAS/Agent | PAT | Endpoint CAN QUERY |

### Dify運用時のポイント

Difyアプリには**固定のPAT**が埋め込まれるため、全ユーザーが同じ権限でアクセスします。

```
User A ─┐                   固定PAT
User B ─┼─▶ Dify App ──────────────▶ Databricks
User C ─┘   （全員同じ権限）         UC権限で制御
```

| 要件 | 推奨方式 |
|------|--------|
| 全員同じデータでOK | Dify + 固定SP（シンプル） |
| 部門別にデータを分けたい | Dify App × SP を分離 |
| ユーザー単位で厳格に制御 | **Databricks Apps**（OAuth U2M） |

> **ベストプラクティス**: Dify用のサービスプリンシパル（SP）を作成し、必要最小限のUC権限をGRANT。  
> 個人PATではなくSPのPATを使うことで、退職・異動の影響を受けない運用が可能。

---
## DSLインポート（オプション）

上記の手動設定の代わりに、DSLファイルからワークフローを復元できます。

1. Dify → **Studio** → **Import DSL File** で以下をインポート:
   - `dsl/02_UC_SQL_API.yml` — UC関数実行ワークフロー
   - `dsl/02_Vector_Search.yml` — Vector Search検索ワークフロー
   - `dsl/02_Genie_API.yml` — Genie自然言語SQLワークフロー
2. インポート後、各ワークフロー内のHTTPノードを確認:
   - **URL**: 自分のworkspace URLに合わせる
   - **UC SQL API**: Body内の `warehouse_id` を自分の環境のWarehouse IDに変更
     （上のセルで表示された `Warehouse ID` をコピー）

> API KeyはDSLに含まれないため、HTTPノードの認証設定でPATを入力してください。

![UC関数実行ワークフロー（SQL API）](../images/02_uc_sql_api.jpg)
*UC関数実行ワークフロー（SQL API）*


![Vector Search検索ワークフロー](../images/02_vector_search.jpg)
*Vector Search検索ワークフロー*


![Genie API自然言語SQLワークフロー](../images/02_genie_api.jpg)
*Genie API自然言語SQLワークフロー*
